# Homework 7: Payables in Python (the inherited notebook)

**ACCTG 6155, Week 7.** Summit Gear Co., accounts payable.

> **Before anything else:** in Colab, choose **File > Save a copy in Drive** and work
> in your copy. The copy that opens from the link does not save your work.

You did receivables in the Lab. This notebook moves to the **payables** side: the
bills Summit Gear owes its vendors. You did not write the first half of this
notebook. A colleague wrote it and handed it to you, so start by reading their code
before you change any of it.

## How to read code you inherited

1. **Read it top to bottom once** before you change anything.
2. **Run it one cell at a time** (Shift+Enter) and read what each cell produces. The
   output tells you what the code actually does.
3. **Ask an AI tool to explain any line you cannot read.** A prompt card is below.
   Read the line yourself first.
4. **Check the numbers** against something you can confirm by eye.

The homework has four parts: **read and analyze**, **debug a real error**, **make a
chart**, and **write a control test**.

**Work the notebook first, then answer the Canvas quiz.** The quiz is one attempt.
Canvas saves your answers if you leave and come back, but nothing counts until you
click Submit, and you cannot reopen it after that. Finishing the notebook first means
you have every number in front of you when you answer.

### Prompt card: get oriented

Copy this into the university AI tool, or ChatGPT or Claude, paste the notebook cell
you are stuck on underneath it, then **read the explanation and check it against what
you see**:

> I am new to Python and pandas. Explain this notebook cell line by line, in
> plain language, as if to an accountant who knows Excel but not Python. Tell
> me what each line produces and why. Do not rewrite it, just explain it.

If you still cannot explain a line in your own words, ask a follow-up question before
you move on.

### Setup

**Colab is the supported path, the same Colab you used in Week 2 and in the Lab.**
pandas and matplotlib are already installed there, so skip to Cell A.

**VS Code with the Jupyter extension also works** if you already have Python on your
computer. Run `%pip install pandas matplotlib` once in a notebook cell. If it errors,
paste the error into the university AI tool and ask what it means, or switch to Colab,
where both libraries are already installed.

---

## The inherited notebook (top portion)

Below is the code you were handed. Run each cell with **Shift+Enter** and read the
output before moving on. The first Canvas questions ask about exactly these cells, so
go slowly here.

In [ ]:
# Cell A - load the libraries and the data
import pandas as pd
import matplotlib.pyplot as plt
print("pandas", pd.__version__)

# Reads the bills file straight from the course data repo, so this runs in
# Colab with no upload. Running locally with the file saved next to this
# notebook? Comment out the URL line and use the local line under it.
BILLS_URL = "https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-fall/week-07/data/hw/vendor_bills.csv"
bills = pd.read_csv(BILLS_URL)
# bills = pd.read_csv("vendor_bills.csv")   # local copy, in the same folder

# .head() shows the first 5 rows so you can eyeball the shape.
bills.head()

In [ ]:
# Cell B - how big is it, and what type is each column?
print("rows, columns:", bills.shape)
print()
print(bills.dtypes)

Look closely at the `dtypes` output from Cell B. The `amount` column is the one you
are about to do math on. **Note what type pandas decided it is.** A number column
shows as `float64` or `int64`. A text column shows as `object` in pandas 2, which is
what Colab runs today, or as `str` in pandas 3. Cell A printed which one you have.
You will be asked about it, and it explains the error in Part 2.

In [ ]:
# Cell C - how many bills fall in each spending category?
# value_counts() tallies the rows per category. No arithmetic here, so this
# works fine even though `amount` has a problem we have not hit yet.
bills["category"].value_counts()

In [ ]:
# Cell D - how many distinct vendors do we owe?
bills["vendor_name"].nunique()

---

## Part 1: Read and analyze

The first quiz questions check that you can read the code you inherited: what it
loads, how big the table is, what each cell returns. Write down your answers as you
go and enter them in the quiz at the end.

Then write two small pieces of **summary analytics** of your own. They are short, and
the Lab showed you the moves. Try them by hand before asking AI.

**1a. Count the bills.** Write one line that prints how many bills are in the table.
The row count is the first number in `bills.shape`, or use `len(bills)`.

In [ ]:
# 1a - your code here: print the number of bills

**1b. Bills per category.** You saw `value_counts()` in Cell C. Write one line, in
your own cell, that produces the count of bills **per category**, and read it off:
which category has the most bills?

In [ ]:
# 1b - your code here: count of bills per category

The **by-group total**, dollars per category, is the natural next step. It needs
arithmetic on `amount`, and that is where the notebook breaks. On to Part 2.

---

## Part 2: Find and fix the error

The cell below is the next thing the original author tried: the **average bill**, by
averaging the `amount` column. **Run it.** It raises an error, and the cell prints the
error's last line for you to read. The last line names the kind of error and gives the
clue.

Leave Cell E as it is. It is set up so the error prints and the rest of the notebook
still runs when you use Run all. Your fix goes in the Part 2 cell further down.

In [ ]:
# Cell E - the average bill amount
# RUN THIS. The average raises an error on this column. The try/except only prints
# that error, so Run all keeps going past this cell. Leave this cell as it is.
try:
    bills["amount"].mean()
except Exception as err:
    print("Cell E raised", type(err).__name__ + ":")
    print(str(err).splitlines()[-1])

### What just happened

If pandas had read the whole `amount` column as numbers, `.mean()` would just work.
It did not, because two cells in `amount` hold **text** instead of a number: someone
typed a note into a number column. A column has one type for every row, so pandas
stored the *entire* column as text, the text type Cell B showed you, and you cannot
average text.

You will see this often in exported data. There are two ways to work out what went
wrong, and you can use either or both.

**Option 1: look at the file yourself.** [Open `vendor_bills.csv`](https://github.com/sean-mccaman/acctg5150-090/blob/main/2026-fall/week-07/data/hw/vendor_bills.csv)
or run the cell below, which lists the rows whose `amount` is not a clean number.
These are the rows your fix has to handle. The fix keeps every row; only these two
amounts become missing values.

In [ ]:
# Option 1 helper - show the rows whose `amount` is NOT a clean number.
def _is_number(x):
    try:
        float(x)
        return True
    except (TypeError, ValueError):
        return False

bad = bills[~bills["amount"].apply(_is_number)]
bad

**Option 2: take the error to AI.** Copy the **last line** of the error and paste it
into your AI tool with this prompt card, then read what it gives back:

> I am using pandas. I ran `bills["amount"].mean()` and got this error:
> *(paste the last line of the error here)*. Explain what this error means and
> why pandas raises it on this column. Do not give me the fix.

### Your fix

Convert `amount` to a real number column. The standard move:

```python
bills["amount"] = pd.to_numeric(bills["amount"], errors="coerce")
```

`errors="coerce"` replaces anything that is not a number with `NaN`, and `.mean()` and
`.sum()` then skip the `NaN` values. Write the fix below, then print the average and
the total.

In [ ]:
# Part 2 - fix the column, then average it and total it.
# Step 1: coerce `amount` to numbers (the bad cells become NaN).
# Step 2: print the average with .mean() and the total with .sum()

# your code here

**Check yourself.** After coercing, two cells are `NaN`, so the total covers 22 of the
24 bills. The cell below should report **2**.

**What the two missing amounts mean.** BILL-1006 and BILL-1014 are unpaid bills from
vendors in this file, so their rows stay. Someone typed a note, `pending` and `see note`, where the amount should
have gone, so the amount was never keyed. The bills are still on the ledger and still
unpaid; only their amounts are missing from this file. Your total after the fix is the
22 bills with a known amount, which is less than Summit Gear actually owes.

**Use your coerced total for the quiz.** [`vendor_bills_clean.csv`](https://github.com/sean-mccaman/acctg5150-090/blob/main/2026-fall/week-07/data/hw/vendor_bills_clean.csv)
has the same 24 bills with the two real amounts filled in, so its total is higher than
yours by the value of those two bills. Open it only to see which rows your fix left out.
The quiz asks for the total `.sum()` prints after your fix, and it asks what that
number measures and what you would do about the two bills before reporting it.

In [ ]:
# how many amounts are missing after coercion? (expect 2)
bills["amount"].isna().sum()

**The by-group total, now that it works.** With `amount` numeric, finish what Part 1
could not: total dollars **per category**, sorted high to low. This is the `groupby`
from Lab 7 Part 2, the PivotTable in one line, now on payables. Read off the category
with the largest total among the 22 bills with a known amount.

In [ ]:
# total amount per category, sorted high to low.
# Hint: group by category, sum the amount, then sort.

# your code here

---

## Part 3: Make a chart

Here is a **ready-made charting function**. You do not write it and you do not change
it. Your job is to hand it data in the **exact shape it expects**, a small DataFrame
with a `category` column and a `total` column, and then call it. Read its docstring
first; it says which columns the function reads.

**Run the cell below first** to define the function, and read its docstring.

In [ ]:
# Cell F - PROVIDED. Run it to define plot_category_totals. Do not edit it.
def plot_category_totals(summary):
    """Draw a bar chart of total payables per spending category.

    Expects a DataFrame named `summary` with exactly two columns:
        category : the spending category (text)
        total    : the total dollar amount for that category (number)
    One row per category. Returns nothing; it draws the chart.
    """
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(summary["category"], summary["total"], color="#BE0000")
    ax.set_title("Summit Gear - total payables by category")
    ax.set_xlabel("category")
    ax.set_ylabel("total amount owed ($)")
    ax.ticklabel_format(axis="y", style="plain")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

### Shape your data, then call it

The function wants a DataFrame with two columns named **`category`** and **`total`**.
Your groupby from Part 2 is *almost* that, but `groupby(...).sum()` returns a
**Series** whose index is the category, not a two-column DataFrame. `.reset_index()`
fixes the shape, and then you rename the amount column to `total`.

### Prompt card: shaping data for a function

> I have a pandas Series from `bills.groupby("category")["amount"].sum()`.
> Explain what `.reset_index()` does to a Series like this, and what the
> columns of the result are named. Do not write the code for me.

Then write the reshape yourself: build the `summary` DataFrame below, then call
`plot_category_totals(summary)`.

In [ ]:
# Part 3 - build a DataFrame called `summary` with columns: category, total
#          (one row per category), then call: plot_category_totals(summary)

# your code here

### Part 3b: the same move, now a count

The totals chart answered *how much* Summit Gear owes per category. Now answer *how
many* bills are in each category. It is the **same reshape with a different
aggregation**: `value_counts()` from Cell C is your count per category. Use the same
`.reset_index()`, and name the second column `count`, which is the column
`plot_category_counts` reads.

**Run the cell below first** to define the second chart function.

In [ ]:
# Cell G - PROVIDED. Run it to define plot_category_counts. Do not edit it.
def plot_category_counts(summary):
    """Draw a bar chart of the NUMBER of bills per spending category.

    Expects a DataFrame (Part 3b builds it as `counts`) with exactly two columns:
        category : the spending category (text)
        count    : how many bills fall in that category (a whole number)
    One row per category. Returns nothing; it draws the chart.
    """
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(summary["category"], summary["count"], color="#1f6f6f")
    ax.set_title("Summit Gear - number of bills by category")
    ax.set_xlabel("category")
    ax.set_ylabel("number of bills")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# Part 3b - build a DataFrame called `counts` with columns: category, count
#           (one row per category), then call: plot_category_counts(counts)
# Hint: bills["category"].value_counts() gives the per-category counts; reshape
#       it into two columns the same way you reshaped the totals.

# your code here

---

## Part 4: A control test for duplicate payments

A bill gets paid twice when two copies of it are entered and approved: the vendor
sends the invoice twice, or it is keyed once from the mail and once from an email.
Audit and AP teams run this kind of test to pick out bills to check by hand.
Once the test is written as a function, next quarter's run is loading the new file and
calling the function again.

In this part you write that test on Summit Gear's full payables file for 2025 and the
first quarter of 2026: about 1,500 bills from 60 vendors.

### Where this test comes from

**The test.** Auditors run a duplicate-payment test on bills a company has already
paid. AP teams can run the same test on bills before they are paid. Audit analytics software
such as IDEA and Arbutus ships it as a built-in duplicates test. The three
conditions below are the common first pass, and each one has a reason:

- **Same vendor** (the same `vendor_id`), because a duplicate payment goes to one
  vendor record twice.
- **Same amount, to the cent**, because a duplicate is usually the same invoice
  keyed twice, from the paper copy and from the emailed copy.
- **Dates a few days apart**, because a resubmitted invoice usually arrives days
  after the first one, while a repeat bill such as monthly rent arrives a month later.
  A short window keeps most monthly bills out of the results. At the end of this part
  you widen it and see what comes in.

**The data.** `ap_bills.csv` is Summit Gear's payables ledger: every bill from 2025
and the first quarter of 2026, with the vendor, the bill date, the amount and the
date it was paid. Summit Gear is the course's company, so the file is synthetic and
built for this class, the same way the Week 3 receivables were; it is not any real
company's data. Like a real ledger, it contains some duplicate payments and some
bills that look like duplicates but are not.
`ap_vendors.csv` is the vendor list, one row per vendor, with its name, category
and payment terms.

### The rule

Two bills are a **suspected duplicate** when all three of these are true:

1. they are to the **same vendor** (`vendor_id`),
2. they are for the **same amount**, to the cent, and
3. their `bill_date` values are **no more than `day_window` days apart**, counting a gap
   of exactly `day_window` days as inside.

The function returns **every bill that pairs with at least one other bill**, both
bills of each pair, each bill once. They are suspected duplicates: someone in AP still
has to look at the invoices and decide which payment, if any, to recover.

### A worked example you can check by hand

| bill_id | vendor_id | bill_date | amount |
|---|---|---|---|
| T-1 | V-1 | 2025-01-02 | 500.00 |
| T-2 | V-1 | 2025-01-05 | 500.00 |
| T-3 | V-1 | 2025-01-20 | 500.00 |
| T-4 | V-2 | 2025-01-03 | 500.00 |
| T-5 | V-2 | 2025-01-04 | 500.01 |

At the default `day_window=5`, the answer is **T-1 and T-2**: same vendor, same
amount, 3 days apart. T-3 is 15 days from T-2. T-4 is a different vendor. T-5 is one
cent off T-4, so it is not the same amount. At `day_window=15`, T-3 joins, because 15
days counts as inside.

### How to build it, one step past the Lab

In Lab 7 you merged the invoices with a *different* table. Here you merge the bills
table **with itself** on `vendor_id` and `amount`. Every row then sits next to every
other bill with the same vendor and amount. Two filters finish the job:

- keep each pair once, where the first `bill_id` is less than the second, since a
  self-merge also pairs every bill with itself and lists each real pair twice;
- keep the pairs whose dates are no more than `day_window` days apart, counting the
  gap either way (the absolute difference in days).

The `bill_id`s that survive, from either side of the pair, are the bills to return.

> **Prompt card:** *"In pandas, what happens when I merge a DataFrame with itself on two
> columns? How do I keep each pair once, and how do I get the number of days between
> two date columns?"* Then write the function yourself.

### The same test in Excel and in SQL

Here is the same test in Excel and in the Week 3 SQL. You do not need to run either
one; they are here so you can see the rule in tools you know. On `ap_bills`, both flag
the same bills your function should return.

**Excel.** With `vendor_id` in column B, `bill_date` in C and `amount` in F, the layout
of `ap_bills.csv`, rows 2 to 1500, one helper column counts the bills that match each row, including itself:

```
=COUNTIFS($B$2:$B$1500, B2, $F$2:$F$1500, F2,
          $C$2:$C$1500, ">=" & (C2 - 5), $C$2:$C$1500, "<=" & (C2 + 5)) > 1
```

`TRUE` means at least one **other** bill has the same vendor and amount within 5 days,
so the row is a suspected duplicate. Both bills of a pair come out `TRUE`.

**SQL, the Week 3 join, with the table joined to itself.** Here `bills` means a table
loaded from `ap_bills.csv`, not the 24-bill extract.

```sql
SELECT DISTINCT b.*
FROM bills AS b
JOIN bills AS o
  ON  o.vendor_id = b.vendor_id
  AND o.amount    = b.amount
  AND o.bill_id  <> b.bill_id
  AND ABS(julianday(o.bill_date) - julianday(b.bill_date)) <= 5
ORDER BY b.vendor_id, b.amount, b.bill_date;
```

The join pairs every bill with every other bill for the same vendor and amount, and
the last condition keeps the pairs inside the window. Your pandas function is the
same idea: the self-merge is the join, and the two filters are the conditions after
`ON`.

**Dollars at risk**, which the quiz asks for, is the amount of the **later** bill in
each flagged pair, judged by `bill_date` (not `paid_date`), added up, with each bill
counted once. For this exercise, treat the later bill as the possible duplicate: it is
the payment AP would look into first.

### Load the payables file

This is the full payables file, a different file from the 24-bill extract you
debugged above, so it gets its own names: `ap_bills` and `ap_vendors`.

In [ ]:
AP_BILLS_URL   = "https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-fall/week-07/data/hw/ap_bills.csv"
AP_VENDORS_URL = "https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-fall/week-07/data/hw/ap_vendors.csv"

ap_bills   = pd.read_csv(AP_BILLS_URL, parse_dates=["bill_date", "paid_date"])
ap_vendors = pd.read_csv(AP_VENDORS_URL)
print("ap_bills  :", ap_bills.shape)
print("ap_vendors:", ap_vendors.shape)
ap_bills.head()

### Write the test

**What `find_duplicate_payments(bills_df, day_window=5)` must do**
- **Takes:** a bills DataFrame with `bill_id`, `vendor_id`, `bill_date` and `amount`,
  and a whole number of days, 5 unless you pass another.
- **Returns:** a DataFrame of every bill in a suspected duplicate pair, both bills of
  each pair, each bill once, with all of the input's columns kept, **sorted by
  `vendor_id`, then `amount`, then `bill_date`**. No suspected duplicates returns an
  empty DataFrame.
- **Docstring:** one sentence, or a short explanation, of what the test flags: same
  vendor, same amount, dates within `day_window` days. Whoever runs the test next
  quarter will likely read the docstring before they read the code.

In [ ]:
def find_duplicate_payments(bills_df, day_window=5):
    """ """
    return None

In [ ]:
# Check yourself on the worked example above (expect ['T-1', 'T-2'], then ['T-1', 'T-2', 'T-3']).
toy_bills = pd.DataFrame({
    "bill_id":   ["T-1", "T-2", "T-3", "T-4", "T-5"],
    "vendor_id": ["V-1", "V-1", "V-1", "V-2", "V-2"],
    "bill_date": pd.to_datetime(["2025-01-02", "2025-01-05", "2025-01-20", "2025-01-03", "2025-01-04"]),
    "amount":    [500.00, 500.00, 500.00, 500.00, 500.01],
})
for window in (5, 15):
    result = find_duplicate_payments(toy_bills, day_window=window)
    print(f"day_window={window}:", None if result is None else sorted(result["bill_id"]))

### Run it on the real file, and hand AP the exception list

The cell below runs your test at the default window, adds the vendor names, and writes
the list to `duplicate_payment_exceptions.csv`, the list AP would review. That file stays
in Colab; you upload only the notebook.

The quiz asks how many bills your test flags and the dollars at risk. Use the empty cell
after the list for the dollars: for each pair, take the bill with the later `bill_date`,
and add those amounts. Adding every flagged row counts both bills of each pair, which
doubles the answer.

In [ ]:
flagged = find_duplicate_payments(ap_bills)
if flagged is not None:
    exceptions = flagged.merge(ap_vendors[["vendor_id", "vendor_name"]], on="vendor_id", how="left")
    print("suspected duplicate bills:", len(exceptions))
    exceptions.to_csv("duplicate_payment_exceptions.csv", index=False)
    display(exceptions[["bill_id", "vendor_id", "vendor_name", "bill_date", "amount", "paid_date"]])

In [ ]:
# Dollars at risk: for each flagged pair, the bill with the later bill_date, added up.
# Hint: each pair shares a vendor_id and an amount.

# your code here

### Why the window exists

Now widen the window to a month. Run the two cells below. The first counts the flagged
bills by vendor; the second lists only the bills that were **not** flagged at 5 days,
with their dates and amounts. Look at what those bills have in common. The quiz asks
about it.

The test has two limits. It cannot see the same
supplier set up under two different vendor ids, which is a frequent cause of duplicate
payments and needs a separate test on the vendor master. And it cannot see a
duplicate that was keyed with a one-cent difference; catching that takes a tolerance,
which also brings in more false alarms.

In [ ]:
wide = find_duplicate_payments(ap_bills, day_window=31)
if wide is not None:
    print("bills flagged at day_window=31:", len(wide))
    print()
    print(wide.merge(ap_vendors, on="vendor_id").groupby(["vendor_id", "vendor_name"])["bill_id"]
              .count().sort_values(ascending=False).head(8))

In [ ]:
# The bills that join at 31 days, with their dates and amounts.
if wide is not None and flagged is not None:
    new_bills = wide[~wide["bill_id"].isin(flagged["bill_id"])]
    new_bills = new_bills.merge(ap_vendors[["vendor_id", "vendor_name"]], on="vendor_id", how="left")
    display(new_bills.sort_values(["vendor_id", "bill_date"])[["bill_id", "vendor_name", "bill_date", "amount"]])

---

## Before you submit

**What is graded in this notebook.** Every item below is checked from the notebook you
upload, so run all the cells and keep their outputs:

| Part | Cell | What has to be there |
|---|---|---|
| 1 | 1a | the number of bills, printed |
| 1 | 1b | the count of bills per category |
| 2 | your fix | `amount` converted to numbers, with the average and the total printed |
| 2 | the groupby | total dollars per category, sorted high to low |
| 3 | `summary` | a two-column DataFrame, and the totals chart drawn from it |
| 3b | `counts` | a two-column DataFrame, and the counts chart drawn from it |
| 4 | `find_duplicate_payments` | a real docstring, and the right bills for the worked example, the real file, and a few cases you do not see |

**Then:**
- **Run the whole notebook once more** (Runtime > Restart session and run all) so the
  saved outputs are real. Cell E prints its error and the run carries on. Both charts
  should appear.
- **Download it:** File > Download > Download .ipynb. Colab names it `hw_07.ipynb` or
  `Copy of hw_07.ipynb`. **Rename it to `<uID>_hw_07.ipynb`**, with your uID (the
  letter u and seven digits) in place of `<uID>`. Upload only the notebook; the
  exception CSV stays in Colab.
- **Open the Canvas quiz, answer the graded questions, and upload the notebook** before
  you click Submit. The Python-basics questions each come with a short snippet: paste
  it into a cell and run it before you answer. The last question, general feedback, is
  optional.